# MIMIC-IV 3.1 aSAH BigQuery Notebook

由 `deepseek.md` 转换生成。SQL 代码块已拆成可运行的 `%%bigquery` 单元；如果当前环境未启用 BigQuery magic，请先运行下方初始化单元。


In [ ]:
# 如当前 notebook 环境提示找不到 %%bigquery，可取消下一行注释后运行一次。
# %load_ext google.cloud.bigquery

感谢您的纠正！MIMIC-IV 3.1版本的表名确实变了。下面给出**修正后的完整代码**，将所有`physionet-data.mimiciv_hosp`替换为`physionet-data.mimiciv_3_1_hosp`，`physionet-data.mimiciv_icu`替换为`physionet-data.mimiciv_3_1_icu`。

---

## 修正后的完整数据提取代码（MIMIC-IV 3.1版）

### 第0步：准备工作


In [ ]:
%%bigquery
-- 目的：创建本研究专用的 BigQuery 数据集 `mimic-study-498508.ash_study`。
-- 原因：后续 cohort 构建会产生多张中间表和最终分析表，统一写入该数据集便于复现、排查和清理。
CREATE SCHEMA IF NOT EXISTS `mimic-study-498508.ash_study`;

-- 验证：返回当前目标数据集名称，确认后续 SQL 使用的目标位置一致。
SELECT
    'mimic-study-498508' AS project_id,
    'ash_study' AS dataset_id,
    '后续 cohort 中间结果和最终分析表均写入该数据集' AS validation_note;


### 第1步：提取aSAH患者队列


In [ ]:
%%bigquery
-- 目的：查看 MIMIC-IV 3.1 诊断字典中与蛛网膜下腔出血相关的 ICD 编码。
-- 原因：正式建 cohort 前先核对编码文本，避免因版本或字典差异导致纳入标准写错。
SELECT DISTINCT
    icd_version,
    icd_code,
    long_title
FROM `physionet-data.mimiciv_3_1_hosp.d_icd_diagnoses`
WHERE LOWER(long_title) LIKE '%subarachnoid hemorrhage%'
   OR LOWER(long_title) LIKE '%subarachnoid haemorrhage%'
ORDER BY icd_version, icd_code;

-- 目的：提取 aSAH 住院记录，并要求同一次住院同时存在 SAH 诊断和动脉瘤诊断。
-- 原因：单纯 I60/430 只能识别蛛网膜下腔出血；增加 ICD-9 437.3 或 ICD-10 I67.1 可提高动脉瘤性 SAH 队列特异性。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.sah_patients` AS
SELECT DISTINCT
    di.subject_id,
    di.hadm_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    a.race,
    a.insurance,
    a.hospital_expire_flag
FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di
INNER JOIN `physionet-data.mimiciv_3_1_hosp.admissions` a
    ON di.subject_id = a.subject_id
   AND di.hadm_id = a.hadm_id
WHERE (
        (di.icd_version = 9 AND di.icd_code LIKE '430%')
     OR (di.icd_version = 10 AND di.icd_code LIKE 'I60%')
)
AND EXISTS (
    SELECT 1
    FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd` di2
    WHERE di2.subject_id = di.subject_id
      AND di2.hadm_id = di.hadm_id
      AND (
            (di2.icd_version = 9 AND di2.icd_code = '437.3')
         OR (di2.icd_version = 10 AND di2.icd_code LIKE 'I67.1%')
      )
);

-- 验证：检查 aSAH 候选队列规模、患者数、住院数、住院内死亡数和重复键数量。
-- 原因：这些指标可快速发现 cohort 为空、重复 hadm_id 或死亡结局字段连接异常。
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT subject_id) AS patient_count,
    COUNT(DISTINCT hadm_id) AS admission_count,
    COUNTIF(hospital_expire_flag = 1) AS in_hospital_deaths,
    COUNT(*) - COUNT(DISTINCT CONCAT(CAST(subject_id AS STRING), '-', CAST(hadm_id AS STRING))) AS duplicate_subject_hadm_rows
FROM `mimic-study-498508.ash_study.sah_patients`;

-- 验证：抽样查看 10 条 cohort 记录，确认入院时间、出院时间和人口学字段非明显错位。
-- 原因：计数只能发现规模问题，样例记录便于人工核对字段含义。
SELECT *
FROM `mimic-study-498508.ash_study.sah_patients`
ORDER BY subject_id, hadm_id
LIMIT 10;


### 第2步：提取首次ICU停留


In [ ]:
%%bigquery
-- 目的：为每个 aSAH 住院记录保留第一次 ICU stay。
-- 原因：一个住院可能包含多个 ICU stay；以首次 ICU stay 作为分析入口可避免同一次住院重复入组。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.first_icu_stay` AS
WITH icu_ranked AS (
    SELECT
        ie.subject_id,
        ie.hadm_id,
        ie.stay_id,
        ie.intime AS icu_intime,
        ie.outtime AS icu_outtime,
        TIMESTAMP_DIFF(ie.outtime, ie.intime, HOUR) AS icu_los_hours,
        ROW_NUMBER() OVER (
            PARTITION BY ie.subject_id, ie.hadm_id
            ORDER BY ie.intime ASC
        ) AS icu_sequence
    FROM `physionet-data.mimiciv_3_1_icu.icustays` ie
    INNER JOIN `mimic-study-498508.ash_study.sah_patients` sp
        ON ie.subject_id = sp.subject_id
       AND ie.hadm_id = sp.hadm_id
)
SELECT
    subject_id,
    hadm_id,
    stay_id,
    icu_intime,
    icu_outtime,
    icu_los_hours
FROM icu_ranked
WHERE icu_sequence = 1;

-- 验证：检查首次 ICU 表的规模、唯一性和 ICU 停留时长分布。
-- 原因：研究要求有 ICU stay，且后续通常会排除 ICU 停留过短者；这里先暴露时长分布便于判断阈值影响。
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT stay_id) AS stay_count,
    COUNT(DISTINCT hadm_id) AS admission_count,
    COUNTIF(icu_los_hours < 24) AS stays_less_than_24h,
    MIN(icu_los_hours) AS min_icu_los_hours,
    APPROX_QUANTILES(icu_los_hours, 4)[OFFSET(2)] AS median_icu_los_hours,
    MAX(icu_los_hours) AS max_icu_los_hours
FROM `mimic-study-498508.ash_study.first_icu_stay`;

-- 验证：确认每个住院最多只保留一个 ICU stay。
-- 原因：如果返回任何行，说明首次 ICU 去重逻辑失败，会导致患者重复进入最终 cohort。
SELECT
    subject_id,
    hadm_id,
    COUNT(*) AS rows_per_admission
FROM `mimic-study-498508.ash_study.first_icu_stay`
GROUP BY subject_id, hadm_id
HAVING COUNT(*) > 1;


### 第3步：提取血红蛋白和贫血时间


In [ ]:
%%bigquery
-- 目的：确认血红蛋白 Hb 在 MIMIC-IV 3.1 labitems 中的 itemid。
-- 原因：后续贫血时间 T0 完全依赖 Hb 测量，先核对 itemid 可避免误提取其他化验项目。
SELECT
    itemid,
    label,
    fluid,
    category
FROM `physionet-data.mimiciv_3_1_hosp.d_labitems`
WHERE LOWER(label) LIKE '%hemoglobin%'
ORDER BY itemid;

-- 目的：提取首次 ICU stay 内所有合理范围内的 Hb 测量。
-- 原因：Hb 序列用于定义首次 Hb<10 和 Hb<7 的时间点；限制在 ICU stay 内可保证时间锚点与重症暴露窗口一致。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.hemoglobin_measurements` AS
SELECT
    le.subject_id,
    le.hadm_id,
    icu.stay_id,
    le.charttime AS hb_time,
    le.valuenum AS hemoglobin
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON le.subject_id = icu.subject_id
   AND le.hadm_id = icu.hadm_id
WHERE le.itemid = 51222
  AND le.valuenum IS NOT NULL
  AND le.valuenum > 3
  AND le.valuenum < 25
  AND le.charttime >= icu.icu_intime
  AND le.charttime <= icu.icu_outtime;

-- 验证：检查 Hb 测量数量、覆盖 stay 数和数值范围。
-- 原因：Hb 覆盖不足会影响 T0 定义，异常范围提示过滤条件或单位可能有问题。
SELECT
    COUNT(*) AS hb_measurement_count,
    COUNT(DISTINCT stay_id) AS stays_with_hb,
    MIN(hemoglobin) AS min_hb,
    APPROX_QUANTILES(hemoglobin, 4)[OFFSET(2)] AS median_hb,
    MAX(hemoglobin) AS max_hb
FROM `mimic-study-498508.ash_study.hemoglobin_measurements`;

-- 目的：找到每次住院首次 Hb<10 的时间，作为宽松输血策略的触发点。
-- 原因：研究设计以 Hb<10 后短时间内输血定义宽松策略，因此必须保存首次达到阈值的时间。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.first_hb_below_10` AS
SELECT
    subject_id,
    hadm_id,
    MIN(hb_time) AS first_below_10_time,
    ARRAY_AGG(hemoglobin ORDER BY hb_time ASC LIMIT 1)[OFFSET(0)] AS hb_at_first_below_10,
    MIN(hemoglobin) AS nadir_hb_below_10
FROM `mimic-study-498508.ash_study.hemoglobin_measurements`
WHERE hemoglobin < 10
GROUP BY subject_id, hadm_id;

-- 验证：检查 Hb<10 触发人群规模和触发点 Hb 范围。
-- 原因：研究排除从未 Hb<10 的患者，该表规模决定后续可进入分析的贫血人群上限。
SELECT
    COUNT(*) AS admissions_with_hb_below_10,
    MIN(hb_at_first_below_10) AS min_first_trigger_hb,
    MAX(hb_at_first_below_10) AS max_first_trigger_hb,
    COUNTIF(hb_at_first_below_10 >= 10) AS invalid_trigger_rows
FROM `mimic-study-498508.ash_study.first_hb_below_10`;

-- 目的：找到每次住院首次 Hb<7 的时间，作为限制输血策略的触发点。
-- 原因：限制策略只在更低 Hb 阈值后输血，保存首次 Hb<7 时间有助于构造互斥治疗分组。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.first_hb_below_7` AS
SELECT
    subject_id,
    hadm_id,
    MIN(hb_time) AS first_below_7_time,
    ARRAY_AGG(hemoglobin ORDER BY hb_time ASC LIMIT 1)[OFFSET(0)] AS hb_at_first_below_7,
    MIN(hemoglobin) AS nadir_hb_below_7
FROM `mimic-study-498508.ash_study.hemoglobin_measurements`
WHERE hemoglobin < 7
GROUP BY subject_id, hadm_id;

-- 验证：检查 Hb<7 触发人群规模和触发点 Hb 范围。
-- 原因：该表用于限制组定义；若触发点 Hb 不低于 7，说明阈值筛选或聚合逻辑错误。
SELECT
    COUNT(*) AS admissions_with_hb_below_7,
    MIN(hb_at_first_below_7) AS min_first_trigger_hb,
    MAX(hb_at_first_below_7) AS max_first_trigger_hb,
    COUNTIF(hb_at_first_below_7 >= 7) AS invalid_trigger_rows
FROM `mimic-study-498508.ash_study.first_hb_below_7`;

-- 目的：创建统一的首次贫血时间表，以首次 Hb<10 作为主要 T0，并保留 Hb 谷值信息。
-- 原因：协变量提取必须发生在 T0 之前；统一 T0 表可保证后续 MAP、乳酸、PF 比值等窗口一致。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.first_anemia_time` AS
SELECT
    b10.subject_id,
    b10.hadm_id,
    b10.first_below_10_time AS first_anemia_time,
    b10.hb_at_first_below_10 AS hb_at_first_anemia,
    b10.nadir_hb_below_10 AS nadir_hb,
    b7.first_below_7_time,
    b7.hb_at_first_below_7,
    b7.nadir_hb_below_7
FROM `mimic-study-498508.ash_study.first_hb_below_10` b10
LEFT JOIN `mimic-study-498508.ash_study.first_hb_below_7` b7
    ON b10.subject_id = b7.subject_id
   AND b10.hadm_id = b7.hadm_id;

-- 验证：检查 T0 表规模、Hb<7 子集数量，以及 T0 是否为空或越界。
-- 原因：T0 是后续所有基线协变量窗口的锚点，任何空值或阈值错误都会造成因果时间顺序错误。
SELECT
    COUNT(*) AS admissions_with_t0,
    COUNTIF(first_below_7_time IS NOT NULL) AS admissions_also_below_7,
    COUNTIF(first_anemia_time IS NULL) AS missing_t0,
    COUNTIF(hb_at_first_anemia >= 10) AS invalid_t0_hb_rows,
    MIN(first_anemia_time) AS earliest_t0,
    MAX(first_anemia_time) AS latest_t0
FROM `mimic-study-498508.ash_study.first_anemia_time`;


### 第4步：提取输血记录

In [ ]:
%%bigquery
-- 目的：确认 ICU inputevents 中红细胞或 PRBC 输注相关 itemid。
-- 原因：输血暴露定义依赖 itemid；先列出候选项便于人工核对是否覆盖红细胞输注项目。
SELECT
    itemid,
    label,
    abbreviation,
    category,
    unitname
FROM `physionet-data.mimiciv_3_1_icu.d_items`
WHERE LOWER(label) LIKE '%red blood cell%'
   OR LOWER(label) LIKE '%prbc%'
ORDER BY itemid;

-- 目的：提取首次 ICU stay 内有效红细胞输血事件。
-- 原因：后续治疗分组需要判断 Hb 阈值后 24 小时内是否发生输血；排除 Rewritten 和非正量记录可减少无效事件。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.transfusions` AS
SELECT
    ie.subject_id,
    ie.hadm_id,
    ie.stay_id,
    ie.starttime AS transfusion_time,
    ie.amount,
    ie.amountuom,
    ie.itemid
FROM `physionet-data.mimiciv_3_1_icu.inputevents` ie
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON ie.subject_id = icu.subject_id
   AND ie.hadm_id = icu.hadm_id
   AND ie.stay_id = icu.stay_id
WHERE ie.itemid IN (225795, 226368)
  AND ie.statusdescription != 'Rewritten'
  AND ie.amount > 0
  AND ie.starttime >= icu.icu_intime
  AND ie.starttime <= icu.icu_outtime;

-- 验证：检查输血事件数量、涉及 stay 数、输血量范围和 itemid 分布。
-- 原因：可确认输血事件没有被过滤为空，也可发现异常单位或异常输血量。
SELECT
    COUNT(*) AS transfusion_event_count,
    COUNT(DISTINCT stay_id) AS stays_with_transfusion,
    MIN(amount) AS min_amount,
    APPROX_QUANTILES(amount, 4)[OFFSET(2)] AS median_amount,
    MAX(amount) AS max_amount,
    STRING_AGG(DISTINCT CAST(itemid AS STRING), ', ' ORDER BY CAST(itemid AS STRING)) AS included_itemids
FROM `mimic-study-498508.ash_study.transfusions`;

-- 验证：抽样查看输血事件，确认时间、单位和数量字段合理。
-- 原因：治疗分组高度依赖输血时间，样例记录便于人工核对事件粒度。
SELECT *
FROM `mimic-study-498508.ash_study.transfusions`
ORDER BY subject_id, hadm_id, transfusion_time
LIMIT 10;


### 第5步：提取生理指标


In [ ]:
%%bigquery
-- 目的：确认 MAP 在 ICU chartevents 中的候选 itemid。
-- 原因：MAP 是 T0 前血流动力学基线协变量，先核对 itemid 可避免误提取非 MAP 指标。
SELECT
    itemid,
    label,
    abbreviation,
    category,
    unitname
FROM `physionet-data.mimiciv_3_1_icu.d_items`
WHERE LOWER(label) LIKE '%mean arterial pressure%'
ORDER BY itemid;

-- 目的：提取 T0 前、首次 ICU stay 内的 MAP 测量。
-- 原因：MAP 可能同时是混杂因子和效应修饰因子；限定在 T0 前可避免纳入暴露后的生理变化。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.map_baseline` AS
SELECT
    ce.subject_id,
    ce.hadm_id,
    ce.stay_id,
    ce.charttime,
    ce.valuenum AS map_value,
    anemia.first_anemia_time
FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON ce.subject_id = icu.subject_id
   AND ce.stay_id = icu.stay_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON ce.subject_id = anemia.subject_id
   AND ce.hadm_id = anemia.hadm_id
WHERE ce.itemid IN (220181, 225309)
  AND ce.valuenum IS NOT NULL
  AND ce.valuenum BETWEEN 30 AND 200
  AND ce.charttime < anemia.first_anemia_time
  AND ce.charttime >= icu.icu_intime;

-- 验证：检查 MAP 覆盖率、数值范围和是否存在 T0 之后记录。
-- 原因：确保基线窗口严格位于 T0 前，并且 MAP 数值没有明显异常。
SELECT
    COUNT(*) AS map_measurement_count,
    COUNT(DISTINCT stay_id) AS stays_with_map,
    MIN(map_value) AS min_map,
    APPROX_QUANTILES(map_value, 4)[OFFSET(2)] AS median_map,
    MAX(map_value) AS max_map,
    COUNTIF(charttime >= first_anemia_time) AS records_on_or_after_t0
FROM `mimic-study-498508.ash_study.map_baseline`;

-- 目的：提取 T0 前、首次 ICU stay 内的乳酸测量。
-- 原因：乳酸反映组织灌注和疾病严重程度，作为基线协变量必须在输血策略判定前获得。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.lactate_baseline` AS
SELECT
    icu.stay_id,
    le.subject_id,
    le.hadm_id,
    le.charttime,
    le.valuenum AS lactate_value,
    anemia.first_anemia_time
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON le.subject_id = icu.subject_id
   AND le.hadm_id = icu.hadm_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON le.subject_id = anemia.subject_id
   AND le.hadm_id = anemia.hadm_id
WHERE le.itemid = 50813
  AND le.valuenum IS NOT NULL
  AND le.valuenum > 0
  AND le.valuenum < 30
  AND le.charttime < anemia.first_anemia_time
  AND le.charttime >= icu.icu_intime;

-- 验证：检查乳酸覆盖率、数值范围和是否存在 T0 之后记录。
-- 原因：乳酸缺失较常见，覆盖率和时间窗验证有助于判断后续缺失值处理压力。
SELECT
    COUNT(*) AS lactate_measurement_count,
    COUNT(DISTINCT stay_id) AS stays_with_lactate,
    MIN(lactate_value) AS min_lactate,
    APPROX_QUANTILES(lactate_value, 4)[OFFSET(2)] AS median_lactate,
    MAX(lactate_value) AS max_lactate,
    COUNTIF(charttime >= first_anemia_time) AS records_on_or_after_t0
FROM `mimic-study-498508.ash_study.lactate_baseline`;

-- 目的：提取 T0 前、首次 ICU stay 内的 PaO2 测量。
-- 原因：PaO2 与 FiO2 匹配后用于计算 PF 比值，反映基线氧合状态。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.pao2_measurements` AS
SELECT
    le.subject_id,
    le.hadm_id,
    le.charttime AS pao2_time,
    le.valuenum AS pao2,
    anemia.first_anemia_time
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON le.subject_id = icu.subject_id
   AND le.hadm_id = icu.hadm_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON le.subject_id = anemia.subject_id
   AND le.hadm_id = anemia.hadm_id
WHERE le.itemid = 50821
  AND le.valuenum IS NOT NULL
  AND le.valuenum BETWEEN 30 AND 700
  AND le.charttime >= icu.icu_intime
  AND le.charttime < anemia.first_anemia_time;

-- 验证：检查 PaO2 覆盖率、数值范围和时间窗。
-- 原因：PF 比值无法在缺少 PaO2 时计算，时间窗错误会引入暴露后信息。
SELECT
    COUNT(*) AS pao2_measurement_count,
    COUNT(DISTINCT hadm_id) AS admissions_with_pao2,
    MIN(pao2) AS min_pao2,
    MAX(pao2) AS max_pao2,
    COUNTIF(pao2_time >= first_anemia_time) AS records_on_or_after_t0
FROM `mimic-study-498508.ash_study.pao2_measurements`;

-- 目的：提取 T0 前、首次 ICU stay 内的 FiO2 测量。
-- 原因：FiO2 是计算 PF 比值的分母；限定 0.21 到 1.0 可统一比例单位并排除明显异常。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.fio2_measurements` AS
SELECT
    ce.subject_id,
    ce.hadm_id,
    ce.stay_id,
    ce.charttime AS fio2_time,
    ce.valuenum AS fio2,
    anemia.first_anemia_time
FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON ce.subject_id = icu.subject_id
   AND ce.stay_id = icu.stay_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON ce.subject_id = anemia.subject_id
   AND ce.hadm_id = anemia.hadm_id
WHERE ce.itemid = 223835
  AND ce.valuenum IS NOT NULL
  AND ce.valuenum BETWEEN 0.21 AND 1.0
  AND ce.charttime >= icu.icu_intime
  AND ce.charttime < anemia.first_anemia_time;

-- 验证：检查 FiO2 覆盖率、数值范围和时间窗。
-- 原因：FiO2 记录常见单位问题；该验证可发现是否存在百分数单位未转换的情况。
SELECT
    COUNT(*) AS fio2_measurement_count,
    COUNT(DISTINCT stay_id) AS stays_with_fio2,
    MIN(fio2) AS min_fio2,
    MAX(fio2) AS max_fio2,
    COUNTIF(fio2_time >= first_anemia_time) AS records_on_or_after_t0
FROM `mimic-study-498508.ash_study.fio2_measurements`;

-- 目的：将每条 PaO2 与其前 4 小时到后 1 小时内最接近的 FiO2 匹配，并计算 PF 比值。
-- 原因：PaO2 和 FiO2 不一定同一时刻记录，采用临床上合理的近邻窗口可提高氧合指标可用性。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.pf_ratio_matched` AS
SELECT
    pa.subject_id,
    pa.hadm_id,
    pa.pao2_time,
    pa.pao2,
    fio2.fio2_time,
    fio2.fio2,
    pa.pao2 / NULLIF(fio2.fio2, 0) AS pf_ratio,
    ABS(TIMESTAMP_DIFF(pa.pao2_time, fio2.fio2_time, MINUTE)) AS time_diff_minutes,
    ROW_NUMBER() OVER (
        PARTITION BY pa.subject_id, pa.hadm_id, pa.pao2_time
        ORDER BY ABS(TIMESTAMP_DIFF(pa.pao2_time, fio2.fio2_time, MINUTE)) ASC
    ) AS rn
FROM `mimic-study-498508.ash_study.pao2_measurements` pa
INNER JOIN `mimic-study-498508.ash_study.fio2_measurements` fio2
    ON pa.subject_id = fio2.subject_id
   AND pa.hadm_id = fio2.hadm_id
WHERE fio2.fio2_time BETWEEN pa.pao2_time - INTERVAL 4 HOUR
                         AND pa.pao2_time + INTERVAL 1 HOUR;

-- 验证：检查 PaO2-FiO2 匹配数量、最近匹配数量和时间差范围。
-- 原因：如果匹配数量很低，PF 比值缺失会较多；时间差异常说明窗口逻辑可能错误。
SELECT
    COUNT(*) AS matched_pair_count,
    COUNTIF(rn = 1) AS nearest_pair_count,
    MIN(time_diff_minutes) AS min_time_diff_minutes,
    MAX(time_diff_minutes) AS max_time_diff_minutes
FROM `mimic-study-498508.ash_study.pf_ratio_matched`;

-- 目的：按住院汇总最低和平均 PF 比值，仅保留每次 PaO2 最近的 FiO2 匹配。
-- 原因：最低 PF 比值代表最差基线氧合状态，后续可作为混杂因子或效应修饰因子。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.min_pf_ratio` AS
SELECT
    subject_id,
    hadm_id,
    MIN(pf_ratio) AS min_pf_ratio,
    AVG(pf_ratio) AS mean_pf_ratio
FROM `mimic-study-498508.ash_study.pf_ratio_matched`
WHERE rn = 1
  AND pf_ratio > 50
  AND pf_ratio < 600
GROUP BY subject_id, hadm_id;

-- 验证：检查 PF 比值汇总覆盖率和范围。
-- 原因：PF 比值范围可提示 FiO2 单位或匹配逻辑异常，覆盖率用于评估最终模型缺失情况。
SELECT
    COUNT(*) AS admissions_with_pf_ratio,
    MIN(min_pf_ratio) AS min_pf_ratio_observed,
    APPROX_QUANTILES(min_pf_ratio, 4)[OFFSET(2)] AS median_min_pf_ratio,
    MAX(min_pf_ratio) AS max_min_pf_ratio
FROM `mimic-study-498508.ash_study.min_pf_ratio`;


### 第6步：提取患者特征


In [ ]:
%%bigquery
-- 目的：提取年龄、性别、种族和入院类型等人口学特征。
-- 原因：这些变量是基础混杂因子，也可能影响输血策略和死亡风险。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.demographics` AS
SELECT
    p.subject_id,
    sp.hadm_id,
    p.anchor_age AS age,
    p.gender,
    a.race,
    a.admission_type
FROM `physionet-data.mimiciv_3_1_hosp.patients` p
INNER JOIN `mimic-study-498508.ash_study.sah_patients` sp
    ON p.subject_id = sp.subject_id
INNER JOIN `physionet-data.mimiciv_3_1_hosp.admissions` a
    ON sp.subject_id = a.subject_id
   AND sp.hadm_id = a.hadm_id
WHERE p.anchor_age >= 18;

-- 验证：检查人口学表规模、年龄范围和是否存在重复住院键。
-- 原因：最终 cohort 需要成年患者，且每个 subject_id/hadm_id 应唯一。
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT subject_id) AS patient_count,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    COUNTIF(age < 18) AS under_18_rows,
    COUNT(*) - COUNT(DISTINCT CONCAT(CAST(subject_id AS STRING), '-', CAST(hadm_id AS STRING))) AS duplicate_subject_hadm_rows
FROM `mimic-study-498508.ash_study.demographics`;

-- 目的：提取首次 ICU stay 内最早一次运动 GCS。
-- 原因：初始神经功能状态是 aSAH 预后的重要混杂因子，使用最早值代表入 ICU 基线状态。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.initial_gcs` AS
WITH gcs_ranked AS (
    SELECT
        ce.subject_id,
        ce.hadm_id,
        ce.stay_id,
        ce.valuenum AS gcs,
        ce.charttime,
        ROW_NUMBER() OVER (
            PARTITION BY ce.stay_id
            ORDER BY ce.charttime ASC
        ) AS rn
    FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
    INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
        ON ce.stay_id = icu.stay_id
    WHERE ce.itemid = 220739
      AND ce.valuenum IS NOT NULL
      AND ce.valuenum BETWEEN 1 AND 6
)
SELECT
    subject_id,
    hadm_id,
    stay_id,
    gcs AS initial_gcs,
    charttime AS initial_gcs_time
FROM gcs_ranked
WHERE rn = 1;

-- 验证：检查初始运动 GCS 覆盖率和取值范围。
-- 原因：itemid 220739 是 GCS motor，合理范围通常为 1 到 6；若出现 15 说明使用了总 GCS 逻辑。
SELECT
    COUNT(*) AS stays_with_initial_gcs,
    MIN(initial_gcs) AS min_initial_gcs,
    MAX(initial_gcs) AS max_initial_gcs,
    COUNTIF(initial_gcs < 1 OR initial_gcs > 6) AS invalid_gcs_rows
FROM `mimic-study-498508.ash_study.initial_gcs`;

-- 目的：汇总 T0 前心率均值、最大值和最小值。
-- 原因：心率反映循环状态和疾病严重程度，需作为 T0 前基线协变量进入模型。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.heart_rate_aggregated` AS
SELECT
    ce.stay_id,
    AVG(ce.valuenum) AS mean_heart_rate,
    MAX(ce.valuenum) AS max_heart_rate,
    MIN(ce.valuenum) AS min_heart_rate
FROM `physionet-data.mimiciv_3_1_icu.chartevents` ce
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON ce.stay_id = icu.stay_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON ce.subject_id = anemia.subject_id
   AND ce.hadm_id = anemia.hadm_id
WHERE ce.itemid = 220045
  AND ce.valuenum IS NOT NULL
  AND ce.valuenum BETWEEN 30 AND 200
  AND ce.charttime < anemia.first_anemia_time
  AND ce.charttime >= icu.icu_intime
GROUP BY ce.stay_id;

-- 验证：检查心率汇总覆盖率和范围。
-- 原因：范围异常提示单位或过滤错误，覆盖率用于评估最终模型缺失。
SELECT
    COUNT(*) AS stays_with_heart_rate,
    MIN(min_heart_rate) AS min_heart_rate_observed,
    MAX(max_heart_rate) AS max_heart_rate_observed,
    AVG(mean_heart_rate) AS average_mean_heart_rate
FROM `mimic-study-498508.ash_study.heart_rate_aggregated`;

-- 目的：按相同 charttime 匹配心率和收缩压，计算 T0 前休克指数临时明细表。
-- 原因：休克指数 = 心率/收缩压，反映循环不稳定程度；先保留明细便于验证匹配质量。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.shock_index_tmp` AS
SELECT
    hr.subject_id,
    hr.hadm_id,
    hr.stay_id,
    hr.charttime,
    hr.valuenum AS heart_rate,
    sbp.valuenum AS sbp,
    hr.valuenum / NULLIF(sbp.valuenum, 0) AS shock_index
FROM `physionet-data.mimiciv_3_1_icu.chartevents` hr
INNER JOIN `physionet-data.mimiciv_3_1_icu.chartevents` sbp
    ON hr.stay_id = sbp.stay_id
   AND hr.charttime = sbp.charttime
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON hr.stay_id = icu.stay_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON hr.subject_id = anemia.subject_id
   AND hr.hadm_id = anemia.hadm_id
WHERE hr.itemid = 220045
  AND sbp.itemid = 220179
  AND hr.valuenum IS NOT NULL
  AND sbp.valuenum IS NOT NULL
  AND hr.valuenum BETWEEN 30 AND 200
  AND sbp.valuenum BETWEEN 50 AND 250
  AND hr.charttime < anemia.first_anemia_time
  AND hr.charttime >= icu.icu_intime;

-- 验证：检查休克指数明细匹配数量和取值范围。
-- 原因：如果严格同一 charttime 匹配导致数量过少，后续可能需要改用近邻匹配策略。
SELECT
    COUNT(*) AS shock_index_pair_count,
    COUNT(DISTINCT stay_id) AS stays_with_shock_index_pairs,
    MIN(shock_index) AS min_shock_index,
    MAX(shock_index) AS max_shock_index
FROM `mimic-study-498508.ash_study.shock_index_tmp`;

-- 目的：按 stay 汇总 T0 前最大和平均休克指数。
-- 原因：最大休克指数可代表基线最差循环状态，后续进入最终 cohort。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.shock_index_aggregated` AS
SELECT
    stay_id,
    MAX(shock_index) AS max_shock_index,
    AVG(shock_index) AS mean_shock_index
FROM `mimic-study-498508.ash_study.shock_index_tmp`
GROUP BY stay_id;

-- 验证：检查休克指数聚合结果规模和范围。
-- 原因：确认每个 stay 至多一行，并确认聚合值无明显异常。
SELECT
    COUNT(*) AS stays_with_shock_index,
    COUNT(DISTINCT stay_id) AS distinct_stays,
    MIN(max_shock_index) AS min_of_max_shock_index,
    MAX(max_shock_index) AS max_of_max_shock_index
FROM `mimic-study-498508.ash_study.shock_index_aggregated`;

-- 目的：汇总 T0 前血小板最低值和平均值。
-- 原因：血小板水平可能影响出血风险、输血决策和预后，需要作为基线协变量。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.platelet_aggregated` AS
SELECT
    icu.stay_id,
    MIN(le.valuenum) AS min_platelet,
    AVG(le.valuenum) AS mean_platelet
FROM `physionet-data.mimiciv_3_1_hosp.labevents` le
INNER JOIN `mimic-study-498508.ash_study.first_icu_stay` icu
    ON le.subject_id = icu.subject_id
   AND le.hadm_id = icu.hadm_id
INNER JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON le.subject_id = anemia.subject_id
   AND le.hadm_id = anemia.hadm_id
WHERE le.itemid = 51265
  AND le.valuenum IS NOT NULL
  AND le.valuenum BETWEEN 10 AND 1000
  AND le.charttime < anemia.first_anemia_time
  AND le.charttime >= icu.icu_intime
GROUP BY icu.stay_id;

-- 验证：检查血小板汇总覆盖率和范围。
-- 原因：血小板异常值会影响模型稳定性，覆盖率用于评估缺失值处理压力。
SELECT
    COUNT(*) AS stays_with_platelet,
    MIN(min_platelet) AS min_platelet_observed,
    APPROX_QUANTILES(min_platelet, 4)[OFFSET(2)] AS median_min_platelet,
    MAX(min_platelet) AS max_min_platelet
FROM `mimic-study-498508.ash_study.platelet_aggregated`;


### 第7步：聚合表创建


In [ ]:
%%bigquery
-- 目的：按 stay 汇总 T0 前 MAP 最低值和平均值。
-- 原因：模型使用聚合后的 MAP 协变量，而不是逐条生命体征明细。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.map_aggregated` AS
SELECT
    stay_id,
    MIN(map_value) AS min_map,
    AVG(map_value) AS mean_map
FROM `mimic-study-498508.ash_study.map_baseline`
GROUP BY stay_id;

-- 验证：检查 MAP 聚合表规模、唯一性和范围。
-- 原因：确认每个 stay 只有一行，且聚合结果仍在合理范围内。
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT stay_id) AS distinct_stays,
    MIN(min_map) AS min_map_observed,
    MAX(mean_map) AS max_mean_map_observed
FROM `mimic-study-498508.ash_study.map_aggregated`;

-- 目的：按 stay 汇总 T0 前乳酸最高值和平均值。
-- 原因：最高乳酸代表基线最差组织灌注状态，平均值代表整体基线水平。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.lactate_aggregated` AS
SELECT
    stay_id,
    MAX(lactate_value) AS max_lactate,
    AVG(lactate_value) AS mean_lactate
FROM `mimic-study-498508.ash_study.lactate_baseline`
GROUP BY stay_id;

-- 验证：检查乳酸聚合表规模、唯一性和范围。
-- 原因：确认每个 stay 只有一行，且聚合结果没有超过前面设定的合理范围。
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT stay_id) AS distinct_stays,
    MIN(mean_lactate) AS min_mean_lactate_observed,
    MAX(max_lactate) AS max_lactate_observed
FROM `mimic-study-498508.ash_study.lactate_aggregated`;


### 第8步：整合最终数据表


In [ ]:
%%bigquery
-- 目的：整合首次 ICU stay、结局、治疗分组和全部基线协变量，生成最终分析 cohort。
-- 原因：因果机器学习模型需要一行代表一个 ICU stay/住院的宽表；仅保留已有输血分组的样本以符合暴露定义。
CREATE OR REPLACE TABLE `mimic-study-498508.ash_study.final_cohort` AS
SELECT
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    sp.hospital_expire_flag AS mortality,
    gcs.initial_gcs,
    tg.transfusion_group,
    d.age,
    d.gender,
    d.race,
    d.admission_type,
    map.min_map,
    map.mean_map,
    lac.max_lactate,
    lac.mean_lactate,
    pf.min_pf_ratio,
    anemia.nadir_hb,
    anemia.first_anemia_time,
    hr.mean_heart_rate,
    si.max_shock_index,
    plt.min_platelet
FROM `mimic-study-498508.ash_study.first_icu_stay` icu
LEFT JOIN `mimic-study-498508.ash_study.sah_patients` sp
    ON icu.subject_id = sp.subject_id
   AND icu.hadm_id = sp.hadm_id
LEFT JOIN `mimic-study-498508.ash_study.initial_gcs` gcs
    ON icu.stay_id = gcs.stay_id
LEFT JOIN `mimic-study-498508.ash_study.demographics` d
    ON icu.subject_id = d.subject_id
   AND icu.hadm_id = d.hadm_id
LEFT JOIN `mimic-study-498508.ash_study.transfusion_groups_corrected` tg
    ON icu.stay_id = tg.stay_id
LEFT JOIN `mimic-study-498508.ash_study.map_aggregated` map
    ON icu.stay_id = map.stay_id
LEFT JOIN `mimic-study-498508.ash_study.lactate_aggregated` lac
    ON icu.stay_id = lac.stay_id
LEFT JOIN `mimic-study-498508.ash_study.min_pf_ratio` pf
    ON icu.subject_id = pf.subject_id
   AND icu.hadm_id = pf.hadm_id
LEFT JOIN `mimic-study-498508.ash_study.first_anemia_time` anemia
    ON icu.subject_id = anemia.subject_id
   AND icu.hadm_id = anemia.hadm_id
LEFT JOIN `mimic-study-498508.ash_study.heart_rate_aggregated` hr
    ON icu.stay_id = hr.stay_id
LEFT JOIN `mimic-study-498508.ash_study.shock_index_aggregated` si
    ON icu.stay_id = si.stay_id
LEFT JOIN `mimic-study-498508.ash_study.platelet_aggregated` plt
    ON icu.stay_id = plt.stay_id
WHERE tg.transfusion_group IS NOT NULL;

-- 验证：检查最终 cohort 规模、唯一性、治疗组分布和死亡率。
-- 原因：最终表是建模输入，必须确认没有重复 stay，且治疗组和结局字段可用。
SELECT
    COUNT(*) AS final_row_count,
    COUNT(DISTINCT stay_id) AS distinct_stays,
    COUNT(*) - COUNT(DISTINCT stay_id) AS duplicate_stay_rows,
    COUNTIF(transfusion_group = 1) AS liberal_group_count,
    COUNTIF(transfusion_group = 0) AS restrictive_group_count,
    AVG(CAST(mortality AS FLOAT64)) AS mortality_rate
FROM `mimic-study-498508.ash_study.final_cohort`;

-- 验证：检查关键协变量非缺失覆盖情况。
-- 原因：建模前需要了解每个变量缺失程度，以决定填补策略或敏感性分析。
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(age IS NOT NULL) AS age_nonmissing,
    COUNTIF(initial_gcs IS NOT NULL) AS initial_gcs_nonmissing,
    COUNTIF(min_map IS NOT NULL) AS min_map_nonmissing,
    COUNTIF(max_lactate IS NOT NULL) AS max_lactate_nonmissing,
    COUNTIF(min_pf_ratio IS NOT NULL) AS min_pf_ratio_nonmissing,
    COUNTIF(nadir_hb IS NOT NULL) AS nadir_hb_nonmissing,
    COUNTIF(mean_heart_rate IS NOT NULL) AS heart_rate_nonmissing,
    COUNTIF(max_shock_index IS NOT NULL) AS shock_index_nonmissing,
    COUNTIF(min_platelet IS NOT NULL) AS platelet_nonmissing
FROM `mimic-study-498508.ash_study.final_cohort`;

-- 验证：抽样查看最终 cohort，便于人工核对字段连接是否错位。
-- 原因：宽表整合最容易发生 join key 错误，样例记录能辅助检查单行结构。
SELECT *
FROM `mimic-study-498508.ash_study.final_cohort`
ORDER BY subject_id, hadm_id, stay_id
LIMIT 10;


### 第9步：验证数据


In [ ]:
%%bigquery
-- 目的：集中复核最终 cohort 的总体样本量和治疗组分布。
-- 验证：该查询用于确认最终样本量、唯一患者/住院/stay 数和治疗组分布。
-- 原因：这是进入 Python 因果建模前的总体验证入口。
SELECT
    COUNT(*) AS total_patients,
    COUNT(DISTINCT subject_id) AS distinct_patients,
    COUNT(DISTINCT hadm_id) AS distinct_admissions,
    COUNT(DISTINCT stay_id) AS distinct_stays,
    COUNTIF(transfusion_group = 1) AS liberal_group,
    COUNTIF(transfusion_group = 0) AS restrictive_group
FROM `mimic-study-498508.ash_study.final_cohort`;

-- 目的：集中复核最终 cohort 的关键变量覆盖情况。
-- 验证：该查询用于确认建模变量的非缺失数量，便于规划填补和敏感性分析。
-- 原因：缺失覆盖率决定后续建模中的填补、变量剔除和敏感性分析策略。
SELECT
    COUNT(*) AS total,
    COUNTIF(min_map IS NOT NULL) AS map_ok,
    COUNTIF(max_lactate IS NOT NULL) AS lactate_ok,
    COUNTIF(min_pf_ratio IS NOT NULL) AS pf_ok,
    COUNTIF(initial_gcs IS NOT NULL) AS gcs_ok,
    COUNTIF(nadir_hb IS NOT NULL) AS nadir_hb_ok,
    COUNTIF(mean_heart_rate IS NOT NULL) AS heart_rate_ok,
    COUNTIF(max_shock_index IS NOT NULL) AS shock_index_ok,
    COUNTIF(min_platelet IS NOT NULL) AS platelet_ok
FROM `mimic-study-498508.ash_study.final_cohort`;

-- 目的：按治疗组查看主要结局和关键连续变量分布。
-- 验证：该查询用于初步比较两组结局率和基线差异，辅助发现混杂或极端值问题。
-- 原因：建模前先看两组基线差异，可帮助判断混杂、阳性假设和极端值问题。
SELECT
    transfusion_group,
    COUNT(*) AS n,
    AVG(CAST(mortality AS FLOAT64)) AS mortality_rate,
    AVG(age) AS mean_age,
    AVG(nadir_hb) AS mean_nadir_hb,
    AVG(min_map) AS mean_min_map,
    AVG(max_lactate) AS mean_max_lactate,
    AVG(min_pf_ratio) AS mean_min_pf_ratio
FROM `mimic-study-498508.ash_study.final_cohort`
GROUP BY transfusion_group
ORDER BY transfusion_group;


---

## 表名对照汇总

| 原表名（MIMIC-IV v2.x）       | 新表名（MIMIC-IV v3.1）           |
| ----------------------------- | --------------------------------- |
| `physionet-data.mimiciv_hosp` | `physionet-data.mimiciv_3_1_hosp` |
| `physionet-data.mimiciv_icu`  | `physionet-data.mimiciv_3_1_icu`  |
| `physionet-data.mimiciv_ed`   | `physionet-data.mimiciv_3_1_ed`   |

---

## Python建模部分

建模代码中的表名不需要修改，因为您只需要导入最终生成的CSV文件。建模代码保持不变。

需要我把Python建模代码也重新发一遍吗（保持原样即可）？
